# 高次元ベイズ最適化：モデル比較と逐次BOベンチマーク

このNotebookでは、robotorchanの高次元入力モデルを**同じ初期観測・同じ有限候補プール・同じ評価予算**で比較します。複数seedのsimple regretの平均とばらつきを可視化し、結果を解釈します。

学ぶ内容：高次元BOの難しさ、PCA/PLS/ランダム射影の違い、`model.fit()`・獲得関数・候補選択の役割、観測追加による逐次更新、simple regretとbest-so-far、seed間の変動です。

**重要：** このベンチマークは有限候補プール上で `qLogExpectedImprovement` を評価します。**連続高次元空間での `optimize_acqf` の探索性能は評価していません**。候補プール全体の真値は合成ベンチマークの診断用であり、モデルの学習には使いません。


## 0. なぜ高次元BOは難しいのか

入力次元が増えると、限られた観測で入力空間を十分に覆うことが難しくなります。GPのカーネルの推定や獲得関数の候補探索も難しくなり得ます。

| モデル | 主な考え方 | 注意点 |
|---|---|---|
| `SingleTaskGP` | 元の全入力を使う基準モデル | 高次元でデータ不足になりやすい |
| `PCAGP` | 入力分散の大きい方向へ圧縮 | 目的関数に重要な方向と一致するとは限らない |
| `PLSGP` | 目的変数との関係を考慮して圧縮 | 少数観測で推定が不安定になることがある |
| `RandomProjectionGP` | ランダムな低次元射影 | 射影の乱数・有効次元に依存 |

この合成問題では入力20次元のうち**目的関数が直接依存するのは最初の5次元**です。したがって、低次元の構造を活用できるかが比較のポイントです。

In [ ]:
from pathlib import Path
import importlib.util
import sys

import matplotlib.pyplot as plt
import pandas as pd

roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
root = next((p for p in roots if (p / 'benchmarks' / 'high_dimensional_bo.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Run from the repository root or examples/notebooks directory')
benchmark_path = root / 'benchmarks' / 'high_dimensional_bo.py'
spec = importlib.util.spec_from_file_location('high_dimensional_bo', benchmark_path)
benchmark = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = benchmark
spec.loader.exec_module(benchmark)


### 合成目的関数の構造を確認する

`benchmark.objective()` は20次元入力を受け取りますが、最初の5列しか目的関数に直接寄与しません。ここでは1列目を変化させ、それ以外を0.5に固定した断面を表示します。

この図は**合成関数の真値**です。実際の最適化では目的関数がどの次元に依存するかは通常未知です。

In [ ]:
import torch

demo_X = torch.full((201, 20), 0.5, dtype=torch.double)
demo_X[:, 0] = torch.linspace(0, 1, 201, dtype=torch.double)
demo_Y = benchmark.objective(demo_X)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(demo_X[:, 0], demo_Y.squeeze(-1))
ax.set(xlabel="First input coordinate x0", ylabel="Objective", title="20D synthetic objective: one-dimensional slice")
plt.show()

## 1. 軽量設定で複数seedのBOを実行

まずは `SingleTaskGP`、`PCAGP`、`PLSGP`、`RandomProjectionGP` の4モデルを比較します。Notebookを短時間で確認できるよう、候補数・反復数・MC sample数は小さめにしています。本格比較では値を増やしてください。

In [ ]:
seeds = [0, 1, 2]
model_names = tuple(benchmark.model_specs(latent_dim=4))
print('Compared models:', model_names)
results = benchmark.run_repeated_benchmark(
    seeds,
    n_initial=10,
    n_candidates=64,
    input_dim=20,
    latent_dim=4,
    n_iterations=5,
    mc_samples=16,
)
summaries = benchmark.aggregate_results(results)
len(results), len(summaries)


### 1回のBO反復で行われること

ベンチマーク内部では、各反復で次の操作を実行します。

1. これまでの観測 `train_X/train_Y` からモデルを構築し、学習する
2. 未評価候補プールでBoTorchの `qLogExpectedImprovement` を評価する
3. 獲得関数が最大の候補を1点選ぶ（`q=1`）
4. その候補の**合成目的関数の観測値**を追加する
5. 評価済み候補をプールから除外して次の反復へ進む

実装の学習処理は `benchmark.fit_model()` が担います。既存のモデルの `make_mll()` とBoTorch `fit_gpytorch_mll()` を利用しており、Notebook内で新しい最適化器は実装しません。

## 2. 生trajectoryと集計結果をDataFrameで確認

生データにはseedが保存されます。モデル比較では平均値だけでなく、同一seedにおけるpairedなtrajectoryも確認できます。

In [ ]:
raw_df = pd.DataFrame([vars(row) for row in results])
summary_df = pd.DataFrame([vars(row) for row in summaries])
assert set(raw_df["model"]) == set(model_names)
assert set(raw_df["seed"]) == set(seeds)
assert raw_df["selected_value"].notna().all()
assert (raw_df["simple_regret"] >= -1e-8).all()
assert (raw_df["n_observations"] >= 10).all()
display(raw_df.head())
display(summary_df.head())


### 結果の列の意味

| 列 | 意味 |
|---|---|
| `model` | 比較した代理モデル |
| `seed` | 初期観測と候補プールを生成した乱数シード |
| `iteration` | 追加評価の回数 |
| `n_observations` | 初期観測を含む累積観測数 |
| `selected_value` | その反復で**実際に評価した候補の値** |
| `best_observed` | これまでの最良実測値 |
| `simple_regret` | 比較対象の既知最良値と最良実測値との差 |

この実装のsimple regretは、**初期観測と固定候補プールを合わせた有限集合の最大値**との差です。連続領域全体の真の大域最適値との差ではありません。

## 3. simple regret trajectoryを可視化

線はseed平均、帯は25%点から75%点です。simple regretは小さいほど、候補プール内の良い点を早く発見できています。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for model, frame in summary_df.groupby('model'):
    frame = frame.sort_values('iteration')
    line = ax.plot(frame['iteration'], frame['simple_regret_mean'], marker='o', label=model)[0]
    ax.fill_between(
        frame['iteration'],
        frame['simple_regret_q25'],
        frame['simple_regret_q75'],
        alpha=0.15,
        color=line.get_color(),
    )
ax.set_xlabel('BO iteration')
ax.set_ylabel('Simple regret')
ax.set_title('高次元BO: simple regret trajectory')
ax.legend()
ax.grid(alpha=0.2)
plt.show()


### best-so-farも同時に確認する

最大化問題では `best_observed` は大きいほど良く、`simple_regret` は小さいほど良い指標です。両者は同じ候補集合の最良値を基準とした表裏の関係ですが、最良値の絶対水準も確認すると解釈しやすくなります。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for model, frame in summary_df.groupby('model'):
    frame = frame.sort_values('iteration')
    ax.plot(frame['iteration'], frame['best_observed_mean'], marker='o', label=model)
ax.set(xlabel="BO iteration", ylabel="Mean best observed value", title="Best-so-far by model")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## 4. seedごとのtrajectoryを確認

平均が同程度でも、初期設計への依存が大きいモデルがあります。次の図では各seedを個別に描き、結果が特定seedだけに支配されていないか確認します。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for (model, seed), frame in raw_df.groupby(['model', 'seed']):
    frame = frame.sort_values('iteration')
    ax.plot(frame['iteration'], frame['simple_regret'], alpha=0.45, label=f'{model} / seed={seed}')
ax.set_xlabel('BO iteration')
ax.set_ylabel('Simple regret')
ax.set_title('seed別 simple regret')
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.2)
plt.show()


### 最終反復時点でモデルを比較する

最終反復の平均・中央値・四分位範囲を表で確認します。**平均だけで優劣を決めず**、ばらつきとseed別のtrajectoryを合わせて判断します。

3 seedは説明用の小規模設定です。統計的な優位性を主張するには不十分です。

In [ ]:
last_iteration = summary_df['iteration'].max()
final_summary = summary_df.loc[summary_df['iteration'] == last_iteration, [
    'model', 'n_seeds', 'simple_regret_mean', 'simple_regret_median',
    'simple_regret_q25', 'simple_regret_q75', 'best_observed_mean',
]].sort_values('simple_regret_mean')
display(final_summary)

## 5. CSVとして保存・再読込

長時間のbenchmarkでは毎回再計算せず、raw trajectoryとsummaryを保存して解析する運用が便利です。

In [ ]:
output_dir = root / 'benchmark_results'
raw_path = output_dir / 'high_dimensional_bo_notebook.csv'
summary_path = output_dir / 'high_dimensional_bo_notebook_summary.csv'
benchmark.write_csv(results, raw_path)
benchmark.write_csv(summaries, summary_path)
loaded_summary = pd.read_csv(summary_path)
loaded_summary.head()


### CSVを保存する際の注意

`write_csv()` は出力先ディレクトリを作成します。保存したCSVには生trajectoryと集計結果を分けて記録します。**再読込したCSVは学習済みGPそのものではありません**。再実験する場合はseed・モデル設定・依存ライブラリのバージョンも保存してください。

## 6. Extendedモデルを比較する場合

AE/VAE、Supervised AE/VAE、Joint系、MAP-SAASも `include_extended=True` で追加できます。Joint/NeuralモデルをBO反復ごとに学習するため計算時間が大きくなります。本格実行時に必要なモデル群だけを比較してください。

```python
extended_results = benchmark.run_repeated_benchmark(
    [0, 1, 2],
    n_initial=12,
    n_candidates=256,
    input_dim=40,
    latent_dim=5,
    n_iterations=10,
    mc_samples=64,
    include_extended=True,
    neural_epochs=20,
    joint_steps=30,
)
```

### Extendedモデルを実行する前に

AE/VAE・Joint系・MAP-SAASは学習時間や初期値の影響が大きくなります。`include_extended=True` はモデルを増やす設定であり、全モデルの計算量を一定にする設定ではありません。

このベンチマークは**候補プールから選ぶ**ため、連続高次元の候補最適化の難しさは比較対象外です。

## 7. モデル選択時の見方

1. 予測benchmarkでRMSE/NLLが極端に悪いモデルを除外します。
2. 学習時間・posterior時間・acquisition評価時間を確認します。
3. 残ったモデルを複数seedのBOで比較し、simple regretの平均とばらつきを確認します。
4. synthetic benchmarkの結果をそのまま実データの順位として扱わず、実データでも同じ比較を行います。

高次元で連続acquisition optimization自体がボトルネックになる場合は、ReducedGPだけではなくREMBO / BAxUS / TuRBOなど探索戦略側の対応も別途検討します。

## 8. 結果の解釈と次の検証

- **高次元の入力を扱えること**と**高次元での連続候補探索が容易なこと**は別問題です。
- PCA/PLS/RandomProjectionのどれが良いかは、目的関数が依存する次元、観測数、ノイズ、射影次元に依存します。
- まず同じseed・同じ候補プール・同じ追加評価回数で比較し、seed間のばらつきも確認します。
- このsimple regretは**有限候補集合内の最良値との差**であり、未知の連続空間全体の真の最適値との差ではありません。
- モデル比較に加え、学習時間、posterior評価時間、獲得関数評価時間も必要に応じて別途測定してください。
- 実務データでは、既知の合成真値を使った診断指標をそのまま利用できません。実際に観測した最良値や、保留した検証データを使うなど評価方法を設計してください。

連続空間での探索戦略がボトルネックになる場合は、TuRBOやREMBOなどの既存探索手法と分けて評価することをおすすめします。